In [1]:
from pathlib import Path
import json

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
from finn.analysis.fpgadataflow.res_estimation import res_estimation
from functools import partial


MODEL_PATH = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/"
    "finn_build_pynqz2/intermediate_models/step_specialize_layers.onnx"
)

FPGA_PART = "xc7z020clg400-1"

model = ModelWrapper(str(MODEL_PATH))

cycles = model.analysis(exp_cycles_per_layer)
resources = model.analysis(partial(res_estimation, fpgapart=FPGA_PART))

print("===================================================")
print("MVAU folding inspection")
print("===================================================")
print("model:", MODEL_PATH)
print()

for node in model.graph.node:
    if node.op_type in ["MVAU_hls", "MVAU_rtl"]:
        inst = getCustomOp(node)

        def get_attr(name):
            try:
                return inst.get_nodeattr(name)
            except Exception:
                return None

        name = node.name
        pe = get_attr("PE")
        simd = get_attr("SIMD")
        mw = get_attr("MW")
        mh = get_attr("MH")

        try:
            exp_cyc = inst.get_exp_cycles()
        except Exception:
            exp_cyc = cycles.get(name, None)

        res = resources.get(name, {})

        print(f"[{name}]")
        print("  op_type :", node.op_type)
        print("  MW      :", mw)
        print("  MH      :", mh)
        print("  PE      :", pe)
        print("  SIMD    :", simd)
        print("  cycles  :", exp_cyc)
        print("  LUT     :", res.get("LUT", None))
        print("  FF      :", res.get("FF", None))
        print("  DSP     :", res.get("DSP", None))
        print("  BRAM_18K:", res.get("BRAM_18K", None))
        print()

MVAU folding inspection
model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_specialize_layers.onnx

[]
  op_type : MVAU_hls
  MW      : 1
  MH      : 16
  PE      : 1
  SIMD    : 1
  cycles  : 2240
  LUT     : 489
  FF      : None
  DSP     : 0
  BRAM_18K: 3

[]
  op_type : MVAU_hls
  MW      : 3
  MH      : 16
  PE      : 1
  SIMD    : 1
  cycles  : 6720
  LUT     : 489
  FF      : None
  DSP     : 0
  BRAM_18K: 3

[]
  op_type : MVAU_hls
  MW      : 48
  MH      : 16
  PE      : 1
  SIMD    : 1
  cycles  : 107520
  LUT     : 489
  FF      : None
  DSP     : 0
  BRAM_18K: 3

[]
  op_type : MVAU_hls
  MW      : 48
  MH      : 16
  PE      : 1
  SIMD    : 1
  cycles  : 107520
  LUT     : 489
  FF      : None
  DSP     : 0
  BRAM_18K: 3

[]
  op_type : MVAU_hls
  MW      : 48
  MH      : 16
  PE      : 1
  SIMD    : 1
  cycles  : 107520
  LUT     : 489
  FF      : None
  DSP     : 0
  BRAM_18K: 3

[]
  op_type : MVAU_hls
  MW 